# W7D3 — Build a RAG Assistant — Guided

**Week 7 · Day 3 · Retrieval, RAG and Recommenders** · Lab

Monday you cut the documents. Tuesday you indexed them. Today the model arrives, and the whole
system is **one string**: an instruction, some retrieved chunks with their ids, the question, and a
format. Four concatenations, and every property you care about is a sentence inside it.

You build that string four times. **V1** is the question alone, and it lies. **V2** adds the
retrieved context and answers correctly — then invents an answer to a question the corpus cannot
support. **V3** adds one sentence and the invention stops. **V4** adds citations that are copied
from chunk metadata rather than composed by the model.

The diff between V2 and V3 is one sentence, and it is the most transferable thing in this week.

**Time budget:** ~115 minutes. Sections 1–2 are the lab; Section 3 is a stretch you may finish at home.

<div dir="rtl" align="right">

# الأسبوع ٧ اليوم ٣ — ابنِ مساعدًا بالتوليد المعزّز بالاسترجاع

**الأسبوع السابع · اليوم الثالث · الاسترجاع والتوليد المعزّز والتوصية** · معمل

يوم الاثنين قطّعت الوثائق، ويوم الثلاثاء فهرستها. واليوم يصل النموذج، والنظام كله **سلسلة نصّية
واحدة**: تعليمة، ومقاطع مسترجَعة بمعرّفاتها، والسؤال، وصيغة الإجابة. أربع وصلات، وكل خاصّية تهمّك
جملةٌ داخلها.

وتبني تلك السلسلة أربع مرّات. فـ**V1** هو السؤال وحده، وهو يكذب. و**V2** يضيف السياق المسترجَع
فيجيب صحيحًا — ثم يخترع إجابةً لسؤال لا تسنده المُدوّنة. و**V3** يضيف جملةً واحدة فيتوقّف الاختراع.
و**V4** يضيف استشهاداتٍ منسوخةً من بيانات المقاطع الوصفية لا مؤلَّفةً من النموذج.

والفرق بين V2 وV3 جملة واحدة، وهو أكثر ما في هذا الأسبوع قابليةً للنقل.

**الزمن المتوقّع:** نحو ١١٥ دقيقة. القسمان الأول والثاني هما المعمل، والقسم الثالث إضافي يمكن إكماله في المنزل.

</div>

> **This is the guided version.** Most of the code is already here. Fill in the lines marked
> `# TODO`. If you want the full challenge, use the `_blank` version instead.

<div dir="rtl" align="right">

> **هذه النسخة الموجَّهة.** معظم الشيفرة موجودة، وعليك إكمال الأسطر المعلَّمة بـ `# TODO`.
> وإذا أردت التحدّي الكامل فاستخدم نسخة `_blank`.

</div>

## Learning objectives

By the end of this lab you can:

- Show, on your own corpus, that a model with no context answers confidently and wrongly.
- Assemble a RAG prompt from retrieved chunks with delimiters and ids, and say what each part does.
- Add a refusal instruction and **measure** that out-of-scope behaviour changed.
- Produce citations that are copied from metadata, and prove no citation was invented.
- Handle two documents that contradict each other without letting the ranking settle it silently.
- Record latency and tokens per question, because tomorrow's evaluation needs both.

<div dir="rtl" align="right">

## أهداف التعلّم

في نهاية هذا المعمل تستطيع:

- أن تُظهر على مُدوّنتك أن نموذجًا بلا سياق يجيب بثقة وبخطأ.
- أن تركّب موجّه RAG من مقاطع مسترجَعة بفواصل ومعرّفات، وأن تقول وظيفة كل جزء.
- أن تضيف تعليمة الرفض وأن **تقيس** أن السلوك خارج النطاق قد تغيّر.
- أن تُنتج استشهادات منسوخة من البيانات الوصفية، وأن تُثبت أن لا استشهاد اختُرع.
- أن تعالج وثيقتين متناقضتين بلا أن يحسم الترتيب الأمر بصمت.
- أن تسجّل الزمن وعدد الرموز لكل سؤال، لأن تقييم الغد يحتاج الاثنين.

</div>

## About the data

**`chroma_store/`** — Tuesday's persisted vector store over the 24 `policy_docs` documents, opened
here with no re-embedding and no re-chunking. If you did not finish Tuesday, `load_artefact` opens
the reference copy from `shared/solutions_cache/`.

**`rag_eval_questions`** — the same 40 questions: 30 answerable with their gold sections, 10
deliberately out of scope. Today you run all 40 through two prompt versions and save what happened.
Tomorrow you score it.

**The model.** Any OpenAI-SDK-compatible endpoint; the course default is a free OpenRouter slug.
The key comes from a `.env` file — **never from a cell in this notebook**, because notebook outputs
get committed and shared.

**Every call is cached to disk under a hash of the exact request.** Re-running a cell costs nothing
and returns the same words. A reference cache ships in `shared/solutions_cache/llm_cache/`, so this
lab completes with **no key and no network** — which is the plan for the afternoon the Wi-Fi fails,
and it always does. If you edit a prompt you are making a new request, and that one needs a key.

**The known problem with this corpus, and today it is a task:** POL-103 §3 says store credit expires
after 12 months and POL-119 §2 says 24. Task 6 asks the question they disagree about.

<div dir="rtl" align="right">

## عن البيانات

**`chroma_store/`** — مخزن المتّجهات المحفوظ من الثلاثاء على وثائق `policy_docs` الأربع والعشرين،
يُفتح هنا بلا إعادة تضمين ولا إعادة تقطيع. وإن لم تُكمل الثلاثاء فإن `load_artefact` يفتح النسخة
المرجعية من `shared/solutions_cache/`.

**`rag_eval_questions`** — الأسئلة الأربعون نفسها: ثلاثون قابلة للإجابة بأقسامها المرجعية، وعشرة
خارج النطاق عمدًا. واليوم تمرّرها كلها في نسختَي موجّه وتحفظ ما حدث. وغدًا تقيسه.

**والنموذج:** أي واجهة متوافقة مع OpenAI، والافتراضي في المقرّر شريحة مجانية من OpenRouter. والمفتاح
من ملف `.env` — **لا من خلية في هذا الدفتر** أبدًا، لأن مخرجات الدفاتر تُرفَع وتُشارَك.

**وكل نداء يُخزَّن على القرص بمفتاح هو بصمة الطلب بالضبط.** فإعادة تشغيل الخلية بلا كلفة وتعيد
الكلمات نفسها. وتُشحن نسخة مرجعية في `shared/solutions_cache/llm_cache/`، فيكتمل هذا المعمل **بلا
مفتاح وبلا شبكة** — وهذه خطّة العصر الذي تسقط فيه الشبكة، وهي تسقط دائمًا. وإن عدّلت موجّهًا فقد
صنعت طلبًا جديدًا، وهذا يحتاج مفتاحًا.

**والمشكلة المعروفة في هذه المُدوّنة، وهي اليوم مهمّة:** تقول `POL-103 §3` إن الرصيد المتجري ينتهي بعد
١٢ شهرًا، وتقول `POL-119 §2` ٢٤. والمهمة السادسة تسأل السؤال الذي يختلفان فيه.

</div>

## Setup

The setup cell prints the cache size before anything else. Two numbers: how many responses you have
locally, and how many are in the shared reference set. If both are zero and you have no key, the
first model call will tell you so in a sentence rather than a stack trace.

<div dir="rtl" align="right">

## الإعداد

تطبع خلية الإعداد حجم المخزن المؤقّت قبل كل شيء. رقمان: كم استجابةً عندك محليًّا، وكم في المجموعة
المرجعية المشتركة. فإن كانا صفرين ولا مفتاح عندك فسيخبرك أول نداء بجملة لا بأثر خطأ.

</div>

In [ ]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, device, versions
from aiep.data import get_dataset, get_dataset_dir, load_artefact
from aiep.llm import chat, cache_stats, DEFAULT_MODEL
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, report
from aiep.viz import use_course_style, savefig

ensure("sentence-transformers", "chromadb", "openai", "python-dotenv", "matplotlib",
       "pandas", "pyarrow")
seed_everything(42)

import json
import re
import textwrap

import chromadb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer

use_course_style()

EMBEDDER = "sentence-transformers/all-MiniLM-L6-v2"
COLLECTION = "policy_chunks"
TOP_K = 3

CORPUS_DIR = get_dataset_dir("policy_docs")
MANIFEST = pd.read_csv(CORPUS_DIR / "manifest.csv").set_index("doc_id")
QUESTIONS = pd.read_parquet(get_dataset("rag_eval_questions"))

STORE = load_artefact("chroma_store")
STORE_COLLECTION = chromadb.PersistentClient(path=str(STORE)).get_collection(COLLECTION)
model = SentenceTransformer(EMBEDDER)

PROMPTS_DIR = ARTEFACT_DIR / "prompts"
PROMPTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"store: {STORE_COLLECTION.count()} chunks at {STORE}")
print(f"questions: {len(QUESTIONS)} — {int(QUESTIONS.answerable.sum())} answerable, "
      f"{int((~QUESTIONS.answerable).sum())} out of scope")
print(f"model: {DEFAULT_MODEL}")
print(f"response cache: {cache_stats()}")
print(versions(), "| device:", device())

## Section 1 — Warm-up: make it lie, on your own corpus  (≈25 min)

One question, asked with **no context at all**:

> How many days do I have to return a digital purchase?

Olo Retail does not exist. Nothing about its return policy is in any model's weights. The model
will still answer, with a number, in confident prose, because producing a plausible continuation is
what it does — this morning's slide 14, on your own data, in one cell.

Then print the true policy from the corpus underneath it. Those two lines next to each other are
the entire motivation for the rest of the week, and they take four seconds to read.

<div dir="rtl" align="right">

## القسم الأول — الإحماء: اجعله يكذب على مُدوّنتك أنت (نحو ٢٥ دقيقة)

سؤال واحد يُسأل **بلا أي سياق**:

> كم يومًا لديّ لإرجاع شراء رقمي؟

شركة Olo Retail لا وجود لها، ولا شيء عن سياسة إرجاعها في أوزان أي نموذج. ومع ذلك سيجيب النموذج
برقم وبنثرٍ واثق، لأن إنتاج استمرار معقول هو ما يفعله — شريحة الصباح ١٤ على بياناتك أنت في خلية
واحدة.

ثم اطبع السياسة الحقيقية من المُدوّنة تحتها. وهذان السطران متجاورين هما دافع بقية الأسبوع كله،
وقراءتهما تستغرق أربع ثوانٍ.

</div>

In [ ]:
V1_QUESTION = "How many days do I have to return a digital purchase?"

V1 = chat(V1_QUESTION, max_tokens=200)

print("V1 — the question, and nothing else")
print("-" * 70)
print(textwrap.fill(V1["text"], 96))
print("-" * 70)
print(f"cached: {V1['cached']} · tokens: {V1['total_tokens']} · "
      f"latency: {V1['latency_s']}s")

TRUTH = (CORPUS_DIR / "docs" / "POL-101.md").read_text(encoding="utf-8")
window = [line for line in TRUTH.splitlines() if "within 14 days" in line][0]
print("\nwhat POL-101 §2 actually says:")
print(textwrap.fill(window.strip(), 96))
print("\nDigital goods are excluded from that window entirely — so the true answer to the "
      "question\nas asked is 'none', and any number at all is wrong.")

WARMUP_INVENTED = bool(re.search(r"\b(\d{1,3})\s*(day|days)\b", V1["text"], re.IGNORECASE))
print(f"\nthe answer contains a specific number of days: {WARMUP_INVENTED}")
print("It was not in the prompt, because the documents were not in the prompt.")

## Section 2 — Core: six tasks  (≈60 min)

1. `retrieve(query, k)` against the store, returning chunks **with** their metadata.
2. **V2** — inject the context. Correct in scope; invents an answer out of scope.
3. **V3** — one sentence, and the out-of-scope behaviour changes. Diff the two prompts.
4. **V4** — citations taken from metadata, verified by hand against the source document.
5. Run all 40 questions through V2 and V4 and save `rag_answers.parquet`.
6. **The contradiction** — two documents disagree, and ranking must not settle it.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: ست مهام (نحو ٦٠ دقيقة)

١. `retrieve(query, k)` على المخزن، تُعيد المقاطع **مع** بياناتها الوصفية.
٢. **V2** — حقن السياق. صحيحٌ داخل النطاق، ويخترع خارجه.
٣. **V3** — جملة واحدة، فيتغيّر السلوك خارج النطاق. قارن الموجّهين.
٤. **V4** — استشهادات مأخوذة من البيانات الوصفية، مُتحقَّقٌ منها يدويًّا مقابل الوثيقة المصدر.
٥. مرّر الأسئلة الأربعين في V2 وV4 واحفظ `rag_answers.parquet`.
٦. **التناقض** — وثيقتان تختلفان، ولا يجوز أن يحسم الترتيب الأمر.

</div>

### Task 2.1 — retrieve, with the metadata attached

`retrieve(query, k)` embeds the query with the store's own model, queries the collection, and
returns a list of chunks each carrying its `id`, its `sections`, its text and its distance.

**The id is the load-bearing part.** It goes into the prompt inside the delimiter, and V4's citation
is that string copied back out. A retriever that returns only text makes citation impossible three
tasks from now, and the fix at that point is to re-ingest the corpus.

<div dir="rtl" align="right">

### المهمة ٢٫١ — الاسترجاع مع البيانات الوصفية

تضمّن `retrieve(query, k)` الاستعلامَ بنموذج المخزن نفسه، وتستعلم المجموعة، وتُعيد قائمة مقاطع يحمل
كلٌّ منها `id` و`sections` ونصّه ومسافته.

**والمعرّف هو الجزء الحامل.** فهو يدخل الموجّه داخل الفاصل، واستشهاد V4 هو تلك السلسلة منسوخةً
عائدة. والمُسترجِع الذي يعيد النصّ وحده يجعل الاستشهاد مستحيلًا بعد ثلاث مهام، وعلاجه حينها إعادة
إدخال المُدوّنة كلها.

</div>

In [ ]:

# TODO: Write retrieve(query, k): embed the query, query the collection, and return the top k chunks as dicts carrying id, sections, text and distance.
# مهمة: اكتب `retrieve(query, k)`: ضمّن الاستعلام، واستعلم المجموعة، وأعِد أفضل k مقاطع قواميسَ فيها المعرّف والأقسام والنصّ والمسافة.

IN_SCOPE = "How many days do I have to return a digital purchase?"
OUT_OF_SCOPE = "What is the warranty period on electronics sold by Olo Retail?"

for label, question in [("in scope", IN_SCOPE), ("out of scope", OUT_OF_SCOPE)]:
    print(f"{label}: {question}")
    for chunk in retrieve(question):
        print(f"   {chunk['id']:<12} {chunk['sections']:<22} d={chunk['distance']:.3f}")
    print()

print("Retrieval returned three chunks for both questions, because retrieval always returns\n"
      "something. That is not a bug and it is not a signal — see task 3.")

### Task 2.2 — V2: inject the context, and watch the good half and the bad half

Build the prompt the way this morning's slide 26 did: the retrieved chunks, each inside a delimiter
carrying its `id`, then the question, then a format instruction. The system message says what the
assistant is. Nothing else happens between retrieval and the model.

Run it twice.

**In scope:** the answer is correct, because both facts it needs are sitting in the prompt. This is
the good half of V2 and it is genuinely good — the system now answers from your documents.

**Out of scope:** the corpus says nothing about warranties. Retrieval still returns three chunks.
V2 answers anyway. Nothing in the prompt told it not to.

**And before anyone proposes a score threshold:** print the distances for both questions. The
out-of-scope question's chunks are not obviously further away. A distance is a measure of word
overlap, not of whether an answer is present, and there is no cut-off that separates those two
cases. That is exactly why the fix has to be an instruction to the model.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — V2: احقن السياق، وشاهد النصف الحسن والنصف السيّئ

ابنِ الموجّه كما في شريحة الصباح ٢٦: المقاطع المسترجَعة، كلٌّ داخل فاصل يحمل `id`، ثم السؤال، ثم
تعليمة الصيغة. وتقول رسالة النظام ما هو المساعد. ولا شيء آخر يحدث بين الاسترجاع والنموذج.

وشغّله مرّتين.

**داخل النطاق:** الإجابة صحيحة، لأن الحقيقتين اللتين يحتاجهما في الموجّه. وهذا هو النصف الحسن من
V2، وهو حسن فعلًا — فالنظام صار يجيب من وثائقك.

**خارج النطاق:** لا تقول المُدوّنة شيئًا عن الضمان. ومع ذلك يعيد الاسترجاع ثلاثة مقاطع، ويجيب V2،
إذ لا شيء في الموجّه يمنعه.

**وقبل أن يقترح أحدٌ عتبةً للدرجة:** اطبع المسافات للسؤالين. فمقاطع السؤال خارج النطاق ليست أبعد
بوضوح. فالمسافة مقياس تداخل كلمات لا مقياس وجود إجابة، ولا حدّ يفصل الحالتين. ولهذا بعينه يجب أن
يكون العلاج تعليمةً للنموذج.

</div>

In [ ]:

SYSTEM_V2 = "You are a policy assistant for Olo Retail. Answer the user's question."
FORMAT_PLAIN = "\nAnswer in at most three sentences."

# TODO: Write build_prompt(question, chunks, fmt) — delimited chunks with their ids, then the question, then the format block.
# مهمة: اكتب `build_prompt(question, chunks, fmt)` — مقاطع بفواصل ومعرّفات، ثم السؤال، ثم كتلة الصيغة.


def ask(question, system, fmt=FORMAT_PLAIN, k=TOP_K, max_tokens=300):
    """Retrieve, build the prompt, call the model, and keep everything that happened."""
    chunks = retrieve(question, k)
    prompt = build_prompt(question, chunks, fmt)
    response = chat(prompt, system=system, max_tokens=max_tokens)
    return {"question": question, "chunks": chunks, "prompt": prompt,
            "system": system, **response}


V2_IN = ask(IN_SCOPE, SYSTEM_V2)
V2_OUT = ask(OUT_OF_SCOPE, SYSTEM_V2)

for label, run in [("IN SCOPE", V2_IN), ("OUT OF SCOPE", V2_OUT)]:
    print(f"=== V2 · {label} ===")
    print(f"Q: {run['question']}")
    print(textwrap.fill(run["text"], 96))
    print(f"retrieved: {[c['sections'] for c in run['chunks']]}")
    print(f"distances: {[round(c['distance'], 3) for c in run['chunks']]}\n")

print("Neither set of distances says 'I cannot answer this'. A threshold on that number would\n"
      "either refuse answerable questions or accept unanswerable ones, and usually both.")

### Task 2.3 — V3: one sentence

Add this to the system message and change **nothing else**:

> Answer using only the provided context. If the context does not contain the answer, say so.

Re-run both questions. In scope: still correct. Out of scope: declined.

Then print the two system messages and diff them, so the size of the change is on screen next to
the size of the effect. It did not make the model more accurate — the in-scope answer was already
right. It gave the system **defined behaviour when the context is insufficient**, which it did not
have before. Undefined behaviour is not "occasionally wrong"; it is "whatever plausible
continuation comes next".

**One note on measuring this.** You need code that decides whether a response *is* a refusal, and
the honest version of that is a keyword check you can read. It is a heuristic, it will
misclassify some answers, and tomorrow's lab calibrates it against hand labels rather than trusting
it. Write it, print what it decides, and read a few outputs yourself.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — V3: جملة واحدة

أضف هذه إلى رسالة النظام ولا تغيّر **شيئًا آخر**:

> أجب من السياق المُعطى وحده. وإن لم يكن الجواب فيه فقل ذلك.

وأعِد تشغيل السؤالين. داخل النطاق: ما يزال صحيحًا. وخارجه: يُرفض.

ثم اطبع رسالتَي النظام وقارنهما، ليكون حجم التغيير على الشاشة بجوار حجم الأثر. فهو لم يجعل النموذج
أدقّ — إذ كانت الإجابة داخل النطاق صحيحة أصلًا. بل أعطى النظام **سلوكًا معرَّفًا حين يكون السياق غير
كافٍ**، ولم يكن له ذلك من قبل. والسلوك غير المعرَّف ليس «مخطئًا أحيانًا» بل «أيّ استمرار معقول يأتي
تاليًا».

**وملاحظة عن قياس هذا:** تحتاج رمزًا يقرّر هل الاستجابة رفضٌ أصلًا، والنسخة الأمينة منه فحصُ كلمات
مفتاحية تستطيع قراءته. وهو تقريبي وسيُخطئ تصنيف بعض الإجابات، ومعمل الغد يعايره بتسميات يدوية بدل
الوثوق به. فاكتبه، واطبع ما يقرّره، واقرأ بعض المخرجات بنفسك.

</div>

In [ ]:

REFUSAL_SENTENCE = (" Answer using only the provided context. If the context does not "
                    "contain the answer, say so.")
SYSTEM_V3 = SYSTEM_V2 + REFUSAL_SENTENCE

REFUSAL_MARKERS = ["does not contain", "doesn't contain", "not contain", "no information",
                   "cannot answer", "can't answer", "not in the provided", "not provided",
                   "does not mention", "doesn't mention", "not specify", "unable to answer",
                   "لا يحتوي", "لا يذكر", "لا توجد معلومات"]


# TODO: Write is_refusal(text) — a readable keyword heuristic, not a model call.
# مهمة: اكتب `is_refusal(text)` — فحصًا تقريبيًّا مقروءًا بالكلمات المفتاحية لا نداءَ نموذج.


V3_IN = ask(IN_SCOPE, SYSTEM_V3)
V3_OUT = ask(OUT_OF_SCOPE, SYSTEM_V3)

print("system message diff, V2 → V3:")
print(f"  V2: {SYSTEM_V2}")
print(f"  V3: {SYSTEM_V2}\033[1m{REFUSAL_SENTENCE}\033[0m")
print(f"  added: {len(REFUSAL_SENTENCE.split())} words, one sentence\n")

RUNS = {"V2 in": V2_IN, "V2 out": V2_OUT, "V3 in": V3_IN, "V3 out": V3_OUT}
for label, run in RUNS.items():
    print(f"=== {label} · refusal detected: {is_refusal(run['text'])}")
    print(textwrap.fill(run["text"], 96), "\n")

REFUSAL_CHANGED = is_refusal(V3_OUT["text"]) and not is_refusal(V2_OUT["text"])
STILL_ANSWERS = not is_refusal(V3_IN["text"])
print(f"out-of-scope behaviour changed: {REFUSAL_CHANGED} · "
      f"in-scope still answered: {STILL_ANSWERS}")

### Task 2.4 — V4: citations that were copied, not composed

Add one clause to the format block: *after each claim, give the id of the document it came from, in
square brackets, exactly as it appears in the `id` attribute.*

The ids are in the prompt already — they came out of chunk metadata on Monday and went into the
delimiter in task 2. So the model is quoting a string you supplied, not producing one. Ask for a
citation **without** supplying ids and it will produce something plausible-looking, because that is
what it does, and a fabricated citation is worse than none: it manufactures the appearance of
verifiability, which is the only thing citations are for.

So verify. Pull the bracketed ids out of the answer with a regex, check every one against the ids
you actually retrieved, and then open one source document and read the sentence yourself.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — V4: استشهادات منسوخة لا مؤلَّفة

أضف بندًا واحدًا إلى كتلة الصيغة: *بعد كل ادّعاء اذكر معرّف الوثيقة التي جاء منها بين قوسين
مربّعين، كما هو في خاصّية `id` تمامًا.*

والمعرّفات في الموجّه أصلًا — خرجت من البيانات الوصفية يوم الاثنين ودخلت الفاصل في المهمة الثانية.
فالنموذج يقتبس سلسلةً أعطيتها إيّاه لا ينشئ واحدة. واطلب استشهادًا **بلا** أن تعطيه معرّفات فسينتج
شيئًا يبدو معقولًا، لأن هذا ما يفعله، والاستشهاد المختلق أسوأ من لا شيء: فهو يصنع مظهر القابلية
للتحقّق، وهي الشيء الوحيد الذي وُجد الاستشهاد له.

فتحقّق. استخرج المعرّفات بين الأقواس من الإجابة بتعبير نمطي، وطابق كلًّا منها مع ما استرجعته فعلًا،
ثم افتح إحدى الوثائق المصدر واقرأ الجملة بنفسك.

</div>

In [ ]:

FORMAT_CITED = (FORMAT_PLAIN + " After each claim, give the id of the document it came from "
                "in square brackets, exactly as it appears in the id attribute.")

# TODO: Write cited_ids(answer) returning every id the answer put in square brackets, and fabricated_ids(answer, chunks) returning the ones that were not retrieved.
# مهمة: اكتب `cited_ids(answer)` تُعيد كل معرّف وضعته الإجابة بين قوسين مربّعين، و `fabricated_ids(answer, chunks)` تُعيد ما لم يُسترجَع منها.

V4_IN = ask(IN_SCOPE, SYSTEM_V3, FORMAT_CITED)

print("=== V4 · in scope, with citations ===")
print(textwrap.fill(V4_IN["text"], 96))
print(f"\nretrieved ids: {[c['sections'] for c in V4_IN['chunks']]}")
print(f"cited ids:     {cited_ids(V4_IN['text'])}")
print(f"fabricated:    {fabricated_ids(V4_IN['text'], V4_IN['chunks'])}")

# Verify one citation by hand: open the document and read the section.
first = (cited_ids(V4_IN["text"]) or [""])[0]
if first:
    doc_id = first.split()[0]
    text = (CORPUS_DIR / "docs" / f"{doc_id}.md").read_text(encoding="utf-8")
    print(f"\n--- {doc_id}, read by hand ---")
    print(textwrap.fill(" ".join(text.split()[:90]), 96))

### Task 2.5 — all 40 questions, both versions

Run every question through **V2** and through **V4**, and save one row per question per version:
question id, version, retrieved ids, answer, cited ids, whether it refused, latency and token count.

Two versions, not one, because tomorrow's evaluation compares them: V2 answers the ten out-of-scope
questions and V4 declines them, and on the thirty answerable ones they should be indistinguishable.
A refusal rate on its own means nothing — a system that refuses everything scores perfectly on
refusals — so the two halves have to be measured together, and that needs both versions on all 40.

Cached responses make this cheap on a re-run. The first run is 80 calls; every run after that is a
disk read.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — الأسئلة الأربعون بالنسختين

مرّر كل سؤال في **V2** وفي **V4**، واحفظ صفًّا لكل سؤال في كل نسخة: معرّف السؤال، والنسخة،
والمعرّفات المسترجَعة، والإجابة، والمعرّفات المُستشهَد بها، وهل رفض، والزمن وعدد الرموز.

نسختان لا واحدة، لأن تقييم الغد يقارنهما: يجيب V2 عن الأسئلة العشرة خارج النطاق ويرفضها V4، وعلى
الثلاثين القابلة للإجابة ينبغي ألّا يتمايزا. ومعدّل الرفض وحده لا يعني شيئًا — فالنظام الذي يرفض كل
شيء يسجّل كمالًا في الرفض — فيجب قياس النصفين معًا، وذلك يحتاج النسختين على الأربعين.

والاستجابات المخزَّنة تجعل هذا رخيصًا عند الإعادة. فأول تشغيل ثمانون نداءً، وكل تشغيل بعده قراءة من
القرص.

</div>

In [ ]:

VERSIONS = {"V2": (SYSTEM_V2, FORMAT_PLAIN),
            "V4": (SYSTEM_V3, FORMAT_CITED)}

# TODO: Run all 40 questions through both versions and build one row per question per version.
# مهمة: مرّر الأسئلة الأربعين في النسختين وابنِ صفًّا لكل سؤال في كل نسخة.

summary = (ANSWERS.groupby(["version", "answerable"])
           .agg(n=("qid", "size"), refused=("refused", "sum"),
                fabricated=("fabricated_ids", lambda s: (s.str.len() > 0).sum())))
print(summary.to_string())

live = ANSWERS[~ANSWERS.cached]
print(f"\n{len(ANSWERS)} rows · {len(live)} live calls this run, "
      f"{len(ANSWERS) - len(live)} from cache")
if len(live):
    print(f"live latency: median {live.latency_s.median():.2f}s, "
          f"p95 {live.latency_s.quantile(0.95):.2f}s · "
          f"tokens: {live.total_tokens.sum():,} total")
print("\nRead the refused column across the two versions before you go on. That table is\n"
      "tomorrow's first metric.")

### Task 2.6 — the contradiction, and why ranking must not settle it

Two documents in this corpus disagree. POL-103 §3 says store credit expires **12 months** after it
is issued; POL-119 §2 says credit converted from loyalty points expires after **24**. Both are
about store credit, both are current, and one of them ranks higher than the other for reasons that
have nothing to do with which policy applies.

Ask the question they disagree about. V4 will cite whichever chunk ranked first and sound exactly
as certain as it does when the documents agree — because nothing told it otherwise, and a distance
of 0.41 against 0.44 is not a judgement about which policy is current.

Add one clause — *if the provided documents disagree, cite both and say they disagree* — re-run,
and show both answers next to each other. Check that the second one cites both documents.

<div dir="rtl" align="right">

### المهمة ٢٫٦ — التناقض، ولماذا لا يجوز أن يحسمه الترتيب

وثيقتان في هذه المُدوّنة تختلفان. تقول `POL-103 §3` إن الرصيد المتجري ينتهي بعد **١٢ شهرًا** من
إصداره، وتقول `POL-119 §2` إن الرصيد المحوَّل من نقاط الولاء ينتهي بعد **٢٤**. وكلتاهما عن الرصيد
المتجري، وكلتاهما سارية، وإحداهما ترتّب أعلى من الأخرى لأسبابٍ لا علاقة لها بأي السياستين تنطبق.

اسأل السؤال الذي تختلفان فيه. سيستشهد V4 بالمقطع الذي رتّب أولًا وسيبدو واثقًا تمامًا كما لو اتّفقت
الوثيقتان — إذ لم يخبره شيء بغير ذلك، ومسافة 0.41 مقابل 0.44 ليست حكمًا على أي السياستين سارية.

أضف بندًا واحدًا — *إن اختلفت الوثائق المُعطاة فاستشهد بكلتيهما وقل إنهما تختلفان* — وأعِد التشغيل،
واعرض الإجابتين متجاورتين. وتحقّق أن الثانية تستشهد بالوثيقتين معًا.

</div>

In [ ]:

CONFLICT_CLAUSE = (" If the provided documents disagree with each other, cite both and say "
                   "that they disagree.")
SYSTEM_V5 = SYSTEM_V3 + CONFLICT_CLAUSE
CONFLICT_QUESTION = QUESTIONS[QUESTIONS.kind == "contradiction"].iloc[0]

# TODO: Ask the contradiction question with V4's prompt and then with the conflict clause added, using enough chunks that both contradicting sections are retrieved.
# مهمة: اسأل سؤال التناقض بموجّه V4 ثم ببند التناقض مضافًا، بعدد مقاطع يكفي لاسترجاع القسمين المتناقضين.

print(f"Q: {CONFLICT_QUESTION.question}")
print(f"the two documents that disagree: {sorted(GOLD_PAIR)}")
print(f"both inside top {CONFLICT_K}: {BOTH_RETRIEVED}\n")

print("=== V4, no conflict instruction ===")
print(textwrap.fill(CONFLICT_BEFORE["text"], 96))
print(f"cited: {cited_ids(CONFLICT_BEFORE['text'])} · cites both: {CITES_BOTH_BEFORE}\n")

print("=== V5, one clause added ===")
print(textwrap.fill(CONFLICT_AFTER["text"], 96))
print(f"cited: {cited_ids(CONFLICT_AFTER['text'])} · cites both: {CITES_BOTH_AFTER}")

## Section 3 — Stretch: the follow-up, and how much context is too much  (≈30 min)

**(a) Query reformulation.** A user asks the follow-up *"what about physical items?"*. That is a
clear thing to say to a person and a useless thing to send to a retriever: it names no policy, no
window and no document, and it fails quietly, because retrieval returns something as always.

Send the conversation so far to the model, ask for a **standalone** version of the question,
retrieve with that, and answer with it. Print the retrieved ids for the raw follow-up and for the
rewrite, side by side. The user never sees the rewrite.

**(b) `top_k`, swept.** Run one question at `k` = 1, 3, 5, 10 and 20 and record the answer, the
token count and the latency at each. More context is not safer: past some point the useful chunks
are a shrinking fraction of a growing prompt, the model has more plausible-but-irrelevant material
to draw on, and you are paying for every token of it. Plot tokens against `k` and read the answers.

<div dir="rtl" align="right">

## القسم الثالث — التوسّع: سؤال المتابعة، وكم من السياق يكون كثيرًا (نحو ٣٠ دقيقة)

**(أ) إعادة صياغة الاستعلام.** يسأل المستخدم متابعةً: *«وماذا عن السلع المادية؟»*. وهذه عبارة واضحة
لإنسان وعديمة الفائدة لمُسترجِع: لا تسمّي سياسة ولا نافذة ولا وثيقة، وتُخفق بصمت، لأن الاسترجاع يعيد
شيئًا كعادته.

أرسل المحادثة حتى الآن إلى النموذج، واطلب صيغةً **مستقلّة** للسؤال، واسترجع بها، وأجب بها. واطبع
المعرّفات المسترجَعة للمتابعة الخام وللصيغة المعاد كتابتها متجاورتين. والمستخدم لا يرى إعادة الصياغة.

**(ب) مسح `top_k`.** شغّل سؤالًا واحدًا عند `k` = ١ و٣ و٥ و١٠ و٢٠، وسجّل الإجابة وعدد الرموز والزمن
عند كلٍّ. فالسياق الأكثر ليس أأمن: بعد حدٍّ ما تصير المقاطع النافعة كسرًا يتقلّص من موجّه يتضخّم،
ويجد النموذج مادّةً معقولةً غير ذات صلة أكثر ليستقي منها، وأنت تدفع ثمن كل رمز. فارسم الرموز مقابل
`k` واقرأ الإجابات.

</div>

In [ ]:

FIRST_TURN = "How many days do I have to return a digital purchase?"
FOLLOW_UP = "what about physical items?"

# TODO: Rewrite the follow-up into a standalone question with a model call, then retrieve with both the raw follow-up and the rewrite and compare what came back.
# مهمة: أعِد صياغة المتابعة سؤالًا مستقلًّا بنداء نموذج، ثم استرجع بالمتابعة الخام وبالصيغة الجديدة وقارن ما عاد.

print(f"raw follow-up : {FOLLOW_UP!r}")
print(f"  retrieved   : {RAW_IDS}")
print(f"rewritten     : {REWRITE!r}")
print(f"  retrieved   : {REWRITTEN_IDS}")
print(f"\nsame chunks: {RAW_IDS == REWRITTEN_IDS}")

In [ ]:

K_VALUES = [1, 3, 5, 10, 20]

# TODO: Sweep top_k for one question and collect k, prompt size, tokens, latency and answer.
# مهمة: امسح `top_k` لسؤال واحد واجمع `k` وحجم الموجّه والرموز والزمن والإجابة.

print(SWEEP[["k", "prompt_chars", "total_tokens", "latency_s", "cited"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(SWEEP.k, SWEEP.prompt_chars, marker="o")
ax.set_xlabel("top_k"); ax.set_ylabel("prompt size (characters)")
ax.set_title("what more context costs, before it helps anything")
savefig(fig, "top_k_cost.png")
plt.show()

for row in SWEEP.itertuples():
    print(f"\n--- k={row.k} ---")
    print(textwrap.fill(row.answer, 96))

**Your sentence.** At which `k` did the answer stop improving, and what were you still paying for
after that? Replace this text.

> …

<div dir="rtl" align="right">

**جملتك.** عند أي `k` توقّفت الإجابة عن التحسّن، وعلامَ كنت تدفع بعد ذلك؟ استبدل هذا النصّ.

> …

</div>

## Save the artefacts

Two things, and tomorrow reads both.

`rag_answers.parquet` — 80 rows, two versions of the system against 40 questions, with the refusal
flag, the citations, the latency and the tokens.

`prompts/` — all five prompt versions as text files, so that tomorrow, and the capstone, can diff
them. A prompt that only exists inside a notebook cell is a prompt nobody can review.

<div dir="rtl" align="right">

## احفظ المُخرَجات

شيئان، والغد يقرأ الاثنين.

`rag_answers.parquet` — ثمانون صفًّا، نسختان من النظام على أربعين سؤالًا، مع علامة الرفض
والاستشهادات والزمن والرموز.

و`prompts/` — نسخ الموجّه الخمس ملفاتِ نصّ، ليقارنها الغد ويقارنها مشروع التخرّج. فالموجّه الذي لا
يوجد إلا داخل خلية دفتر موجّهٌ لا يستطيع أحد مراجعته.

</div>

In [ ]:
ARTEFACT_DIR.mkdir(parents=True, exist_ok=True)
ANSWERS.to_parquet(ARTEFACT_DIR / "rag_answers.parquet", index=False)

PROMPT_FILES = {
    "V1_no_context.txt": f"(no system message)\n\n{V1_QUESTION}\n",
    "V2_context.txt": f"SYSTEM:\n{SYSTEM_V2}\n\nUSER:\n{V2_IN['prompt']}\n",
    "V3_refusal.txt": f"SYSTEM:\n{SYSTEM_V3}\n\nUSER:\n{V3_IN['prompt']}\n",
    "V4_citations.txt": f"SYSTEM:\n{SYSTEM_V3}\n\nUSER:\n{V4_IN['prompt']}\n",
    "V5_conflict.txt": f"SYSTEM:\n{SYSTEM_V5}\n\nUSER:\n{CONFLICT_AFTER['prompt']}\n",
}
for name, text in PROMPT_FILES.items():
    (PROMPTS_DIR / name).write_text(text, encoding="utf-8")

print(f"rag_answers.parquet — {len(ANSWERS)} rows")
print(f"prompts/ — {len(PROMPT_FILES)} files at {PROMPTS_DIR}")
print(f"llm cache now holds {cache_stats()['local']} local responses")
print("\nthe one-sentence diffs, in order:")
print(f"  V2 → V3: {REFUSAL_SENTENCE.strip()}")
print(f"  V3 → V4: {FORMAT_CITED[len(FORMAT_PLAIN):].strip()}")
print(f"  V4 → V5: {CONFLICT_CLAUSE.strip()}")

## Sanity check

<div dir="rtl" align="right">

## فحص سلامة

</div>

In [ ]:
check(WARMUP_INVENTED and "excluded" in window.lower() or WARMUP_INVENTED,
      f"V1 must answer the no-context question with a specific number of days — it gave: "
      f"{V1['text'][:120]!r}. The corpus says digital goods are excluded from the window "
      f"entirely, so any number is invented, and that is the demonstration",
      f"يجب أن يجيب V1 عن السؤال بلا سياق بعدد أيام محدّد — وقد أعطى: {V1['text'][:120]!r}. "
      f"والمُدوّنة تقول إن السلع الرقمية مستثناة من النافذة كليًّا، فأي رقم مخترع، وهذا هو العرض")

check(REFUSAL_CHANGED and STILL_ANSWERS,
      f"one sentence must change the out-of-scope behaviour and leave the in-scope answer alone — "
      f"V2 out-of-scope refused: {is_refusal(V2_OUT['text'])}, V3 out-of-scope refused: "
      f"{is_refusal(V3_OUT['text'])}, V3 in-scope answered: {STILL_ANSWERS}. Both halves matter: "
      f"a system that refuses everything is not an improvement",
      f"يجب أن تغيّر جملة واحدة السلوك خارج النطاق وأن تترك الإجابة داخله كما هي — رفض V2 خارج "
      f"النطاق: {is_refusal(V2_OUT['text'])}، ورفض V3: {is_refusal(V3_OUT['text'])}، وأجاب V3 "
      f"داخل النطاق: {STILL_ANSWERS}. والنصفان يهمّان: فالنظام الذي يرفض كل شيء ليس تحسينًا")

FABRICATED = ANSWERS[ANSWERS.fabricated_ids.str.len() > 0]
check(len(FABRICATED) == 0,
      f"every cited id must exist in that question's retrieved set — {len(FABRICATED)} of "
      f"{len(ANSWERS)} answers cite something that was never retrieved: "
      f"{FABRICATED[['qid', 'version', 'fabricated_ids']].head(3).to_dict('records')}. A "
      f"fabricated citation is worse than none, because it manufactures verifiability",
      f"يجب أن يكون كل معرّف مُستشهَد به في المجموعة المسترجَعة لذلك السؤال — و{len(FABRICATED)} من "
      f"{len(ANSWERS)} إجابة تستشهد بما لم يُسترجَع قط. والاستشهاد المختلق أسوأ من لا شيء، لأنه "
      f"يصنع قابلية تحقّق زائفة")

check(len(ANSWERS) == 2 * len(QUESTIONS)
      and ANSWERS.refused.notna().all()
      and ANSWERS.latency_s.notna().all() and ANSWERS.total_tokens.notna().all(),
      f"rag_answers.parquet must hold both versions of all {len(QUESTIONS)} questions with a "
      f"refusal flag, a latency and a token count on every row — got {len(ANSWERS)} rows, "
      f"{int(ANSWERS.refused.isna().sum())} missing flags, "
      f"{int(ANSWERS.total_tokens.isna().sum())} missing token counts",
      f"يجب أن يحوي `rag_answers.parquet` نسختَي النظام على الأسئلة {len(QUESTIONS)} كلها، مع علامة "
      f"رفض وزمن وعدد رموز في كل صفّ — والناتج {len(ANSWERS)} صفًّا")

V2_OOS = ANSWERS[(ANSWERS.version == "V2") & (~ANSWERS.answerable)].refused.mean()
V4_OOS = ANSWERS[(ANSWERS.version == "V4") & (~ANSWERS.answerable)].refused.mean()
check(V4_OOS > V2_OOS,
      f"V4 must refuse more of the out-of-scope questions than V2 — V2 refused {V2_OOS:.0%}, V4 "
      f"refused {V4_OOS:.0%}. This is the same one sentence measured over ten questions instead "
      f"of one, and tomorrow scores it properly",
      f"يجب أن يرفض V4 من الأسئلة خارج النطاق أكثر ممّا يرفض V2 — رفض V2 {V2_OOS:.0%} ورفض V4 "
      f"{V4_OOS:.0%}. وهذه هي الجملة نفسها مقيسةً على عشرة أسئلة بدل سؤال، والغد يقيسها كما ينبغي")

check(BOTH_RETRIEVED and CITES_BOTH_AFTER,
      f"the conflict case must retrieve both contradicting sections (got {BOTH_RETRIEVED}) and, "
      f"once the conflict clause is added, cite both (got {CITES_BOTH_AFTER}; before the clause "
      f"it cited {cited_ids(CONFLICT_BEFORE['text'])}). Ranking is word overlap, and it must not "
      f"be what settles which policy is current",
      f"يجب أن تسترجع حالة التناقض القسمين المتناقضين (والناتج {BOTH_RETRIEVED})، وأن تستشهد "
      f"بكليهما بعد إضافة بند التناقض (والناتج {CITES_BOTH_AFTER}). فالترتيب تداخل كلمات، ولا يجوز "
      f"أن يكون هو ما يحسم أي سياسة سارية")

check(all((PROMPTS_DIR / name).exists() for name in PROMPT_FILES),
      f"all five prompt versions must be on disk in prompts/ so they can be diffed — found "
      f"{sorted(p.name for p in PROMPTS_DIR.iterdir())}",
      f"يجب أن تكون نسخ الموجّه الخمس على القرص في `prompts/` لتُقارَن — والموجود "
      f"{sorted(p.name for p in PROMPTS_DIR.iterdir())}")

report()

## What's next

**Tomorrow you find out whether any of today's instructions were followed.** None of them are
guaranteed — they were asked for. D4 computes three metrics separately (context recall,
faithfulness, answer relevance), calibrates the judge that produces two of them against your own
hand labels, and puts V2 and V4 in one table so the refusal rate is read next to the answerable
scores rather than on its own.

Then it fixes the worst five questions and measures again, which is the part that turns today's
demo into a system with known properties.

**Capstone M3 is due tomorrow**, and it is graded on exactly that discipline.

<div dir="rtl" align="right">

## ما التالي

**غدًا تعرف هل اتُّبعت تعليمات اليوم أصلًا.** فلا شيء منها مضمون — إنما طُلب طلبًا. ويحسب اليوم
الرابع ثلاثة مقاييس منفصلة (استدعاء السياق، والأمانة للمصدر، وصلة الإجابة)، ويعاير الحَكَم الذي
يُنتج اثنين منها بتسمياتك اليدوية، ويضع V2 وV4 في جدول واحد ليُقرأ معدّل الرفض بجوار درجات الأسئلة
القابلة للإجابة لا وحده.

ثم يُصلح أسوأ خمسة أسئلة ويقيس ثانيةً، وهذا هو الجزء الذي يحوّل عرض اليوم إلى نظام معروف الخصائص.

**ويُسلَّم إنجاز مشروع التخرّج الثالث غدًا**، وهو مُقيَّم على هذا الانضباط بعينه.

</div>